## Funnel analysis

The cells below will investigate answers to the main business questions on the user funnel in our synthetic ecommerce store data: 
- How many sessions reach each stage?
- Where is the biggest drop-off?
- How does conversion vary by device/source/country?
- Which segment has unusually strong/weak conversion?


In [ ]:
# Import Python packages
from snowflake.snowpark.context import get_active_session
import pandas as pd

# Get the current credentials
session = get_active_session()

# Load the view and create a pandas dataframe 
df = session.table("PRODUCT_ANALYTICS.ANALYTICS.FCT_SESSIONS").to_pandas()
df.describe()

In [ ]:
funnel = {
    "Page View": df["HAS_PAGE_VIEW"].sum(),
    "Add to Cart": df["HAS_ADD_TO_CART"].sum(),
    "Checkout": df["HAS_CHECKOUT"].sum(),
    "Purchase": df["HAS_PURCHASE"].sum()
}

funnel_df = pd.DataFrame(
    funnel.items(),
    columns=["stage", "sessions"]
)

funnel_df["conversion_from_previous"] = (
    funnel_df["sessions"]
    / funnel_df["sessions"].shift(1)
)
funnel_df


The funnel shows the proportion of sessions progressing through each stage of the purchase journey.

- **Page View → Add to Cart: 67.9%**
  Approximately 68% of sessions with a page view progressed to adding a product to the cart.

- **Add to Cart → Checkout: 55.1%**
  About 55% of sessions that added a product proceeded to checkout. This is the largest drop-off in the funnel, suggesting that the transition from cart to checkout is a potential area for investigation.

- **Checkout → Purchase: 74.8%**
  Nearly 75% of checkout sessions resulted in a purchase, indicating relatively strong completion once users reach checkout.

Overall, the funnel suggests that the largest opportunity for improvement is between **Add to Cart and Checkout**, rather than at the final checkout-to-purchase stage.

In [ ]:
overall_conversion = (
    funnel_df.loc[
        funnel_df["stage"] == "Purchase",
        "sessions"
    ].iloc[0]
    /
    funnel_df.loc[
        funnel_df["stage"] == "Page View",
        "sessions"
    ].iloc[0]
)
print(f"Overall Page View → Purchase conversion is {round(overall_conversion * 100, 2)}%",)

In [ ]:
test = df.groupby("DEVICE").agg(
    sessions=("SESSION_ID", "count"),
    purchases=("HAS_PURCHASE", "sum")
).assign(
    conversion=lambda x: x["purchases"] / x["sessions"]
)
print(test)

In [ ]:
test = df.groupby("SOURCE").agg(
    sessions=("SESSION_ID", "count"),
    purchases=("HAS_PURCHASE", "sum")
).assign(
    conversion=lambda x: x["purchases"] / x["sessions"]
)
print(test)

In [ ]:
test = df.groupby("COUNTRY").agg(
    sessions=("SESSION_ID", "count"),
    purchases=("HAS_PURCHASE", "sum")
).assign(
    conversion=lambda x: x["purchases"] / x["sessions"]
)
print(test)